<a href="https://colab.research.google.com/github/sergio7654/ppsspp-ffmpeg/blob/master/notebooks/microWakeWord_train_any_wakeword.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# microWakeWord — Train any wake word

A single-notebook trainer for custom wake words on ESPHome `micro_wake_word` devices
(M5Stack Atom Echo, Voice PE, etc).

Two modes:
- **`generate`** — Piper TTS generates ~30k positive samples + your confusables in Colab.
  Easiest path. Requires an IPA pronunciation of your wake word.
- **`bundle`** — You upload a zip with your own samples (real recordings, ElevenLabs
  voices, accent-matched TTS). Higher quality but you do the prep work.

## Required runtime
**Runtime → Change runtime type → A100 GPU + High-RAM**.
T4 OOMs during validation. A100 + High-RAM gives 40 GB VRAM + 85 GB system RAM.

## What you get
A `<output_name>.tflite` (~60 KB) + companion `.json` manifest, ready to drop into
your ESPHome config under `micro_wake_word: models:`.

## Workflow
1. Edit the **CONFIGURE HERE** cell below for your wake word
2. **Runtime → Run all**, walk away ~45 minutes
3. Find `<output_name>.tflite` + `.json` in your Drive folder when it's done
4. Test on hardware — likely needs manifest tuning (cutoff, sliding_window) for
   your specific model. See the deployment notes at the end.

Built from working production deployment of "Hey Harold" — all known
upstream bugs are patched in this notebook.


In [1]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE YOUR WAKE WORD HERE                       ║
# ╚═══════════════════════════════════════════════════════════════════════╝

# ─── Wake word identity ───
WAKE_WORD = "Hola Sofia"           # Human-readable name (shown in HA)
OUTPUT_NAME = "hola_sofia"         # Filename (no spaces, lowercase)
AUTHOR = "Sergio"               # Goes in the manifest
AUTHOR_WEBSITE = ""

# ─── Mode ───
MODE = "generate"   # "generate" (Piper makes samples) | "bundle" (you uploaded a zip)

# ─── Drive folder (created if missing) ───
DRIVE_FOLDER = "wakeword_training_hola_sofia"

# ─── If MODE == "generate" ───
# Look up IPA for your wake word: https://www.internationalphoneticassociation.org/IPA-chart
# or use eSpeak: `espeak-ng -q --ipa "Hey Harold"` on Linux
WAKE_WORD_IPA_US = "ˈola soˈfi.a"
WAKE_WORD_IPA_UK = "ˈola soˈfi.a"
SAMPLES_US = 30000                  # ~12 min on T4, ~7 min on A100
SAMPLES_UK = 0                  # 0 to skip UK pass

# Confusable phrases — should NOT trigger your wake word.
# Include: phonetic neighbors, the bare name without prefix, common
# false-trigger phrases like other assistant names.
CONFUSABLE_PHRASES = [
    # Muy parecidas acústicamente
    "hola Sonia",
    "hola Silvia",
    "hola Olivia",
    "hola Lucía",
    "hola María",
    "hola Rocío",
    "hola Lucía",
    "hola señoría",

    # Misma estructura
    "hola familia",
    "hola amiga",
    "hola cariño",
    "hola bonita",
    "hola buenos días",

    # Contienen Sofía pero NO son el wake word
    "oye Sofía",
    "eh Sofía",
]
SAMPLES_PER_CONFUSABLE = 1000   # 500 = light, 1000 = recommended, 2000 = max

# ─── If MODE == "bundle" ───
# Expected: <DRIVE_FOLDER>/data_bundle.zip with this layout:
#   generated_samples/        positive samples (.wav, 16 kHz mono)
#   real_recordings/          (optional) real mic recordings
#   confusable_negatives/     (optional) hard negative samples
BUNDLE_NAME = "data_bundle.zip"

# ─── Manifest tuning (works well as defaults; tune after on-device testing) ───
PROBABILITY_CUTOFF = 0.85          # 0.5 = too lenient, 0.97 = okay_nabu strict
SLIDING_WINDOW_SIZE = 5            # 5 = okay_nabu default; higher = slower fire
TENSOR_ARENA_SIZE = 50000          # 30000 works, 50000 has more headroom

# ─── Languages this model is trained for (manifest metadata) ───
TRAINED_LANGUAGES = ["es"]         # add "it", "es" etc if you have multi-accent samples

# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                           END OF CONFIG                                ║
# ╚═══════════════════════════════════════════════════════════════════════╝
print(f"Training '{WAKE_WORD}' as {OUTPUT_NAME} in mode={MODE!r}")
print(f"Output → /content/drive/MyDrive/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite")


Training 'Hola Sofia' as hola_sofia in mode='generate'
Output → /content/drive/MyDrive/wakeword_training_hola_sofia/hola_sofia.tflite


In [21]:
import os
import shutil
import time

source = "/content/wakeword_training/positive_16k"
zip_base = "/content/positive_16k"

print("Comprimiendo archivos...", flush=True)
start = time.time()

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir="/content/wakeword_training",
    base_dir="positive_16k"
)

print("ZIP creado:", zip_path)
print("Tamaño:", round(os.path.getsize(zip_path) / 1024**2, 1), "MB")
print("Tiempo:", round(time.time() - start, 1), "segundos")

destination = "/content/drive/MyDrive/wakeword_training_hola_sofia/positive_16k.zip"

print("Copiando a Google Drive...", flush=True)
shutil.copy2(zip_path, destination)

print("Copia terminada")
print("Tamaño en Drive:", round(os.path.getsize(destination) / 1024**2, 1), "MB")

Comprimiendo archivos...
ZIP creado: /content/positive_16k.zip
Tamaño: 650.0 MB
Tiempo: 26.4 segundos
Copiando a Google Drive...
Copia terminada
Tamaño en Drive: 650.0 MB


In [9]:
# === Mount Drive ===
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive folder: {DRIVE_DIR}')
if MODE == 'bundle':
    BUNDLE_PATH = f'{DRIVE_DIR}/{BUNDLE_NAME}'
    assert os.path.exists(BUNDLE_PATH), (
        f'MODE=bundle but {BUNDLE_PATH} does not exist. Upload your data zip there.')
    print(f'Found bundle: {os.path.getsize(BUNDLE_PATH)/1024/1024:.1f} MB')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive folder: /content/drive/MyDrive/wakeword_training_hola_sofia


In [17]:
import os
import shutil
import time

source = "/content/drive/MyDrive/wakeword_training_hola_sofia/positive_samples.zip"
destination = "/content/positive_samples.zip"

print("Existe:", os.path.isfile(source))

if os.path.isfile(source):
    start = time.time()
    shutil.copyfile(source, destination)

    print("ZIP copiado correctamente")
    print("Tamaño:", round(os.path.getsize(destination) / 1024**2, 1), "MB")
    print("Tiempo:", round(time.time() - start, 1), "segundos")
else:
    print("Archivos disponibles en la carpeta:")
    print(os.listdir("/content/drive/MyDrive/wakeword_training_hola_sofia"))

Existe: True
ZIP copiado correctamente
Tamaño: 948.8 MB
Tiempo: 14.4 segundos


In [3]:
# === Install microWakeWord (kernel-restart-free) ===
# Workarounds for two upstream bugs:
#  1. kahrendt/microWakeWord setup.py has no find_packages() — non-editable
#     install skips the audio/ subpackage. Editable install needs kernel
#     restart, breaks Run All. Fix: install deps + sys.path.insert().
#  2. train.py calls .numpy() on values that newer TF returns as numpy
#     arrays already. Patch with hasattr() guard.
import os, sys, subprocess, importlib, re

DEPS = [
    'audiomentations', 'audio_metadata', 'datasets', 'mmap_ninja', 'numpy',
    'pymicro-features', 'pyyaml', 'tensorflow>=2.16', 'webrtcvad-wheels',
    'ai-edge-litert',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)

if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kahrendt/microWakeWord'], check=True)

MWW_DIR = '/content/microWakeWord'
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()

fp = '/content/microWakeWord/microwakeword/train.py'
src = open(fp).read()
patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src
)
n = patched.count('hasattr') - src.count('hasattr')
if n > 0:
    open(fp, 'w').write(patched)
    print(f'Patched {n} .numpy() calls in train.py')

import microwakeword
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print('OK: microwakeword.audio.* imports clean')


Installing dependencies...
OK: microwakeword.audio.* imports clean


In [2]:
# === Data preparation ===
# Mode-aware: either unzip user's bundle, or generate samples inline via Piper.
import os, zipfile

os.chdir('/content')

if MODE == 'bundle':
    print(f'Extracting {BUNDLE_PATH}...')
    with zipfile.ZipFile(BUNDLE_PATH, 'r') as zf:
        zf.extractall('/content')
    for d in ['generated_samples', 'real_recordings', 'confusable_negatives']:
        p = f'/content/{d}'
        if os.path.exists(p):
            n = sum(1 for _ in os.scandir(p) if _.name.endswith('.wav'))
            print(f'  {d}: {n} WAVs')
        else:
            print(f'  {d}: MISSING (will train without)')

elif MODE == 'generate':
    # Defer to the Piper sample-gen cells below
    print('MODE=generate — Piper sample-gen cells will produce samples')

else:
    raise ValueError(f'Unknown MODE: {MODE!r}. Use "bundle" or "generate".')


MODE=generate — Piper sample-gen cells will produce samples


In [3]:
# === Piper sample generator install (Spanish) ===
if MODE == 'generate':
    import glob, os, shutil, subprocess, sys, urllib.request

    PIPER_REPO_DIR = '/content/piper'
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'

    subprocess.run(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], check=True)

    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'pip', 'setuptools', 'wheel', 'cython'
    ], check=True)

    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'piper-tts', 'piper-sample-generator'
    ], check=True)

    if not os.path.exists(PIPER_REPO_DIR):
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/rhasspy/piper',
            PIPER_REPO_DIR
        ], check=True)

    if not os.path.exists(PIPER_SAMPLE_GENERATOR_DIR):
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/rhasspy/piper-sample-generator',
            PIPER_SAMPLE_GENERATOR_DIR
        ], check=True)

    PIPER_PYTHON_DIR = f'{PIPER_REPO_DIR}/src/python'
    MA_DIR = f'{PIPER_PYTHON_DIR}/piper_train/vits/monotonic_align'
    MA_IMPORT_DIR = f'{MA_DIR}/monotonic_align'
    MA_BUILD_DIR = f'{MA_DIR}/piper_train/vits/monotonic_align'

    shutil.rmtree(f'{PIPER_PYTHON_DIR}/build', ignore_errors=True)
    shutil.rmtree(MA_IMPORT_DIR, ignore_errors=True)
    shutil.rmtree(f'{MA_DIR}/piper_train', ignore_errors=True)

    os.makedirs(MA_IMPORT_DIR, exist_ok=True)
    os.makedirs(MA_BUILD_DIR, exist_ok=True)

    open(f'{MA_IMPORT_DIR}/__init__.py', 'a').close()

    subprocess.run(
        f'cd {MA_DIR} && {sys.executable} setup.py build_ext --inplace',
        shell=True,
        check=True
    )

    built = next(iter(glob.glob(f'{MA_BUILD_DIR}/core.*')), None)
    assert built, 'monotonic_align core extension build failed'

    shutil.copy2(built, MA_IMPORT_DIR)

    for path in (PIPER_PYTHON_DIR, PIPER_SAMPLE_GENERATOR_DIR):
        if path not in sys.path:
            sys.path.insert(0, path)

    # Spanish Piper voice
    os.makedirs('models', exist_ok=True)

    MODEL_PATH = 'models/es_ES-sharvard-medium.onnx'
    MODEL_CONFIG_PATH = f'{MODEL_PATH}.json'

    if not os.path.exists(MODEL_PATH):
        print('Downloading Spanish es_ES sharvard model...')

        urllib.request.urlretrieve(
            'https://huggingface.co/rhasspy/piper-voices/resolve/main/es/es_ES/sharvard/medium/es_ES-sharvard-medium.onnx',
            MODEL_PATH
        )

        urllib.request.urlretrieve(
            'https://huggingface.co/rhasspy/piper-voices/resolve/main/es/es_ES/sharvard/medium/es_ES-sharvard-medium.onnx.json',
            MODEL_CONFIG_PATH
        )

    print('Piper ready')
    print('Model:', MODEL_PATH)

else:
    print('Skipped (MODE != generate)')

Piper ready
Model: models/es_ES-sharvard-medium.onnx


In [ ]:
!cd /content/piper-sample-generator && python -m piper_sample_generator --help

usage: __main__.py [-h] --max-samples MAX_SAMPLES --model MODEL
                   [--batch-size BATCH_SIZE]
                   [--slerp-weights SLERP_WEIGHTS [SLERP_WEIGHTS ...]]
                   [--length-scales LENGTH_SCALES [LENGTH_SCALES ...]]
                   [--noise-scales NOISE_SCALES [NOISE_SCALES ...]]
                   [--noise-scale-ws NOISE_SCALE_WS [NOISE_SCALE_WS ...]]
                   [--output-dir OUTPUT_DIR] [--max-speakers MAX_SPEAKERS]
                   [--phoneme-input] [--verbose]
                   text

positional arguments:
  text

options:
  -h, --help            show this help message and exit
  --max-samples MAX_SAMPLES
                        Maximum number of samples to generate
  --model MODEL         Path to PyTorch generator (.pt) or Piper voice model
                        (.onnx)
  --batch-size BATCH_SIZE
                        CUDA batch size (generator only)
  --slerp-weights SLERP_WEIGHTS [SLERP_WEIGHTS ...]
                        Speak

In [ ]:
# === Generate positive samples (Spanish) ===
if MODE == 'generate':
    import os, subprocess, sys

    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    MODEL_PATH = '/content/models/es_ES-sharvard-medium.onnx'
    PIPER_BATCH = 256

    def run_piper(target_word, max_samples, output_dir):
        cmd = [
            sys.executable, '-m', 'piper_sample_generator',
            target_word,
            '--phoneme-input',
            '--model', MODEL_PATH,
            '--max-samples', str(max_samples),
            '--batch-size', str(PIPER_BATCH),
            '--noise-scales', '0.5',
            '--noise-scale-ws', '0.6',
            '--output-dir', output_dir
        ]

        subprocess.run(
            cmd,
            cwd=PIPER_SAMPLE_GENERATOR_DIR,
            check=True
        )

    os.makedirs(
        f'{PIPER_SAMPLE_GENERATOR_DIR}/generated_samples',
        exist_ok=True
    )

    print(f'Generating {SAMPLES_US} Spanish samples for {WAKE_WORD_IPA_US!r}...')

    run_piper(
        WAKE_WORD_IPA_US,
        SAMPLES_US,
        'generated_samples'
    )

    n = sum(
        1 for f in os.listdir(
            f'{PIPER_SAMPLE_GENERATOR_DIR}/generated_samples'
        )
        if f.endswith('.wav')
    )

    print(f'Total positive samples: {n}')

else:
    print('Skipped (MODE != generate)')


Generating 30000 Spanish samples for 'ˈola soˈfi.a'...


In [27]:
import os
import importlib.util

print("Modelo Piper:",
      os.path.isfile("/content/models/es_ES-sharvard-medium.onnx"))

print("Generador:",
      os.path.isdir("/content/piper-sample-generator"))

print("Módulo:",
      importlib.util.find_spec("piper_sample_generator"))

Modelo Piper: True
Generador: True
Módulo: ModuleSpec(name='piper_sample_generator', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7dd0ff93ce30>, origin='/content/piper-sample-generator/piper_sample_generator/__init__.py', submodule_search_locations=['/content/piper-sample-generator/piper_sample_generator'])


In [28]:
# === Generate confusable negatives (Spanish) ===
if MODE == 'generate':
    import os, subprocess, sys
    from pathlib import Path

    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    MODEL_PATH = '/content/models/es_ES-sharvard-medium.onnx'

    os.makedirs('confusable_negatives', exist_ok=True)

    for phrase in CONFUSABLE_PHRASES:
        safe = (
            phrase.replace(' ', '_')
                  .replace(',', '')
                  .replace("'", '')
        )

        existing = len(
            list(Path('confusable_negatives').glob(f'{safe}_*.wav'))
        )

        if existing >= SAMPLES_PER_CONFUSABLE:
            print(f'  {phrase!r}: {existing} already, skip')
            continue

        tmp = f'/tmp/confusable_{safe}'
        os.makedirs(tmp, exist_ok=True)

        print(
            f'  generating {SAMPLES_PER_CONFUSABLE} '
            f'for {phrase!r}...'
        )

        subprocess.run([
            sys.executable, '-m', 'piper_sample_generator',
            phrase,
            '--model', MODEL_PATH,
            '--max-samples', str(SAMPLES_PER_CONFUSABLE),
            '--batch-size', '256',
            '--noise-scales', '0.5',
            '--noise-scale-ws', '0.6',
            '--output-dir', tmp
        ],
        cwd=PIPER_SAMPLE_GENERATOR_DIR,
        check=True)

        # Move + rename with safe prefix
        for f in os.listdir(tmp):
            if f.endswith('.wav'):
                os.rename(
                    f'{tmp}/{f}',
                    f'confusable_negatives/{safe}_{f}'
                )

    n = sum(
        1 for f in os.listdir('confusable_negatives')
        if f.endswith('.wav')
    )

    print(f'Total confusable negatives: {n}')

else:
    print('Skipped (MODE != generate)')

  generating 1000 for 'hola Sonia'...
  generating 1000 for 'hola Silvia'...
  generating 1000 for 'hola Olivia'...
  generating 1000 for 'hola Lucía'...
  generating 1000 for 'hola María'...
  generating 1000 for 'hola Rocío'...
  'hola Lucía': 1000 already, skip
  generating 1000 for 'hola señoría'...
  generating 1000 for 'hola familia'...
  generating 1000 for 'hola amiga'...
  generating 1000 for 'hola cariño'...
  generating 1000 for 'hola bonita'...
  generating 1000 for 'hola buenos días'...
  generating 1000 for 'oye Sofía'...
  generating 1000 for 'eh Sofía'...
Total confusable negatives: 14000


In [29]:
import os
import shutil
import time

source = os.path.abspath("confusable_negatives")
zip_base = "/content/confusable_negatives"

start = time.time()

print("Comprimiendo negativos...", flush=True)

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=os.path.dirname(source),
    base_dir=os.path.basename(source)
)

print("ZIP creado:", zip_path)
print("Tamaño:", round(os.path.getsize(zip_path) / 1024**2, 1), "MB")
print("Tiempo:", round(time.time() - start, 1), "segundos")

destination = "/content/drive/MyDrive/wakeword_training_hola_sofia/confusable_negatives.zip"

print("Copiando a Google Drive...", flush=True)
shutil.copy2(zip_path, destination)

print("Guardado correctamente en Google Drive")

Comprimiendo negativos...
ZIP creado: /content/confusable_negatives.zip
Tamaño: 433.9 MB
Tiempo: 18.1 segundos
Copiando a Google Drive...
Guardado correctamente en Google Drive


In [31]:
# === Download microWakeWord negative datasets ===

import os
import zipfile
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

REPO = "kahrendt/microwakeword"
BASE = "/content/negative_datasets"

os.makedirs(BASE, exist_ok=True)

datasets = [
    "speech.zip",
    "dinner_party.zip",
    "no_speech.zip",
    "dinner_party_eval.zip",
    "speech_background.zip",
    "dinner_party_background.zip",
    "no_speech_background.zip",
]

for filename in datasets:
    print(f"\nProcesando {filename}...", flush=True)

    path = hf_hub_download(
        repo_id=REPO,
        repo_type="dataset",
        filename=filename,
        local_dir=BASE
    )

    destination = os.path.join(
        BASE,
        filename.replace(".zip", "")
    )

    marker = os.path.join(destination, ".extraction_complete")

    if os.path.exists(marker):
        print("Ya extraído, omitiendo")
        continue

    os.makedirs(destination, exist_ok=True)

    with zipfile.ZipFile(path) as z:
        z.extractall(destination)

    with open(marker, "w") as f:
        f.write("OK")

    print("Extraído en:", destination)

print("\nDatasets descargados y extraídos")

=== ARCHIVOS DISPONIBLES ===
Total archivos: 9
dinner_party.zip
dinner_party_background.zip
dinner_party_eval.zip
no_speech.zip
no_speech_background.zip
speech.zip
speech_background.zip

=== COMPROBACIÓN DE URL ===
HTTP: 404
URL final: https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/fma_16k.tar
Content-Type: text/plain; charset=utf-8


In [4]:
# === Download microWakeWord negative datasets ===

import os
import zipfile
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

REPO = "kahrendt/microwakeword"
BASE = "/content/negative_datasets"

os.makedirs(BASE, exist_ok=True)

datasets = [
    "speech.zip",
    "dinner_party.zip",
    "no_speech.zip",
    "dinner_party_eval.zip",
    "speech_background.zip",
    "dinner_party_background.zip",
    "no_speech_background.zip",
]

for filename in datasets:
    print(f"\nProcesando {filename}...", flush=True)

    path = hf_hub_download(
        repo_id=REPO,
        repo_type="dataset",
        filename=filename,
        local_dir=BASE
    )

    destination = os.path.join(
        BASE,
        filename.replace(".zip", "")
    )

    marker = os.path.join(destination, ".extraction_complete")

    if os.path.exists(marker):
        print("Ya extraído, omitiendo")
        continue

    os.makedirs(destination, exist_ok=True)

    with zipfile.ZipFile(path) as z:
        z.extractall(destination)

    with open(marker, "w") as f:
        f.write("OK")

    print("Extraído en:", destination)

print("\nDatasets descargados y extraídos")


Procesando speech.zip...


speech.zip: reconstructing file:   0%|          |  0.00B / 3.18GB            

speech.zip: downloading bytes:           |  0.00B            

Extraído en: /content/negative_datasets/speech

Procesando dinner_party.zip...


dinner_party.zip: reconstructing file:   0%|          |  0.00B /  444MB            

dinner_party.zip: downloading bytes:           |  0.00B            

Extraído en: /content/negative_datasets/dinner_party

Procesando no_speech.zip...


no_speech.zip: reconstructing file:   0%|          |  0.00B / 2.00GB            

no_speech.zip: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

In [18]:
%pip install -U audiomentations

  Attempting uninstall: audiomentations
    Found existing installation: audiomentations 0.33.0
    Uninstalling audiomentations-0.33.0:
      Successfully uninstalled audiomentations-0.33.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [audiomentations]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
piper-sample-generator 3.2.0 requires audiomentations==0.33.0, but you have audiomentations 0.43.1 which is incompatible.


In [11]:
from pathlib import Path
import shutil

origen = Path("/content/confusable_features")
destino = Path("/content/drive/MyDrive/wakeword_training_hola_sofia")
destino.mkdir(parents=True, exist_ok=True)

archivo = destino / "confusable_features.zip"

if not origen.is_dir():
    raise FileNotFoundError(f"No existe {origen}")

for conjunto in ["training", "validation", "testing"]:
    carpeta = origen / conjunto / "wakeword_mmap"
    if not carpeta.is_dir() or not any(carpeta.iterdir()):
        raise RuntimeError(f"Faltan características en {carpeta}")

print("Comprimiendo espectrogramas negativos...")

shutil.make_archive(
    "/content/confusable_features",
    "zip",
    root_dir="/content",
    base_dir="confusable_features"
)

print("Copiando a Google Drive...")
shutil.copy2("/content/confusable_features.zip", archivo)

print("Copia completada")
print("Archivo:", archivo)
print("Tamaño:", round(archivo.stat().st_size / 1024**3, 2), "GB")

Comprimiendo espectrogramas negativos...
Copiando a Google Drive...
Copia completada
Archivo: /content/drive/MyDrive/wakeword_training_hola_sofia/confusable_features.zip
Tamaño: 1.12 GB


In [6]:
import os

for carpeta in [
    "/content/mit_rirs",
    "/content/negative_datasets",
    "/content/fma_16k",
    "/content/audioset_16k",
]:
    existe = os.path.isdir(carpeta)
    wavs = 0
    if existe:
        wavs = sum(
            f.lower().endswith(".wav")
            for _, _, files in os.walk(carpeta)
            for f in files
        )

    print(f"{carpeta}: existe={existe}, WAV={wavs}")

Mounted at /content/drive

Copiando positive_16k.zip a Colab...
Descomprimiendo positive_16k.zip...
Terminado: positive_16k.zip

Copiando confusable_negatives.zip a Colab...
Descomprimiendo confusable_negatives.zip...
Terminado: confusable_negatives.zip

=== COMPROBACIÓN ===
/content/wakeword_training/positive_16k → 30000 WAV
/content/confusable_negatives → 14000 WAV


In [ ]:
from microwakeword.audio.augmentation import Augmentation

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15,
        'TanhDistortion': 0.10,
        'PitchShift': 0.15,
        'BandStopFilter': 0.10,
        'AddColorNoise': 0.20,
        'AddBackgroundNoise': 0.0,
        'Gain': 1.00,
        'GainTransition': 0.25,
        'RIR': 0.60,
    },
    impulse_paths=['mit_rirs'],
    background_paths=[],
    background_min_snr_db=-5,
    background_max_snr_db=20,
    min_jitter_s=0.10,
    max_jitter_s=0.50,
)

print("Augmentation inicializado correctamente")

In [ ]:
# === Augmentation + feature extraction ===
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
import os, shutil, traceback
from mmap_ninja.ragged import RaggedMmap

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15, 'TanhDistortion': 0.10,
        'PitchShift': 0.15, 'BandStopFilter': 0.10,
        'AddColorNoise': 0.20, 'AddBackgroundNoise': 0.85,
        'Gain': 1.00, 'GainTransition': 0.25, 'RIR': 0.60,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5, background_max_snr_db=20,
    min_jitter_s=0.10, max_jitter_s=0.50,
)

os.makedirs('generated_augmented_features', exist_ok=True)
SPLIT_CONFIG = {
    'training':   {'split_name': 'train',      'repetition': 3, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
}

for split, cfg in SPLIT_CONFIG.items():
    out = f'generated_augmented_features/{split}'
    mmap = f'{out}/wakeword_mmap'
    if os.path.exists(mmap) and list(os.scandir(mmap)):
        print(f'{split}: cached, skipping')
        continue
    if os.path.exists(mmap):
        shutil.rmtree(mmap)
    os.makedirs(out, exist_ok=True)
    print(f'Generating {split} (rep={cfg["repetition"]}, slide={cfg["slide_frames"]})...')
    try:
        sg = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    except Exception:
        traceback.print_exc()
        if os.path.exists(mmap): shutil.rmtree(mmap)
        raise
print('Positive features ready')

# Confusable features (only if confusable_negatives/ exists)
if os.path.exists('confusable_negatives') and os.listdir('confusable_negatives'):
    print('Generating confusable features...')
    confusable_clips = Clips(
        input_directory='confusable_negatives', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('confusable_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'confusable_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=confusable_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Confusable features ready')

# Real recording features (only if real_recordings/ exists)
if os.path.exists('real_recordings') and os.listdir('real_recordings'):
    print('Generating real-recording features...')
    real_clips = Clips(
        input_directory='real_recordings', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('real_recording_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'real_recording_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=real_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Real-recording features ready')


In [12]:
from pathlib import Path

print("Directorio actual:", Path.cwd())
print("OUTPUT_NAME:", globals().get("OUTPUT_NAME", "NO DEFINIDO"))

rutas = [
    "generated_augmented_features",
    "confusable_features",
    "negative_datasets/speech",
    "negative_datasets/dinner_party",
    "negative_datasets/no_speech",
    "negative_datasets/dinner_party_eval",
]

for ruta in rutas:
    carpeta = Path(ruta)
    print(f"\n{ruta}")
    print("  Existe:", carpeta.is_dir())

    if carpeta.is_dir():
        for split in ["training", "validation", "testing"]:
            subcarpeta = carpeta / split
            print(f"  {split}: {subcarpeta.exists()}")

        ejemplos = list(carpeta.rglob("*.ninja"))
        print("  Archivos .ninja:", len(ejemplos))
        if ejemplos:
            print("  Ejemplo:", ejemplos[0])

Directorio actual: /content
OUTPUT_NAME: hola_sofia

generated_augmented_features
  Existe: True
  training: True
  validation: True
  testing: True
  Archivos .ninja: 141
  Ejemplo: generated_augmented_features/testing/wakeword_mmap/dtype.ninja

confusable_features
  Existe: True
  training: True
  validation: True
  testing: True
  Archivos .ninja: 141
  Ejemplo: confusable_features/testing/wakeword_mmap/dtype.ninja

negative_datasets/speech
  Existe: True
  training: False
  validation: False
  testing: False
  Archivos .ninja: 141
  Ejemplo: negative_datasets/speech/speech/training/voices_lav_clo_training_mmap/dtype.ninja

negative_datasets/dinner_party
  Existe: True
  training: False
  validation: False
  testing: False
  Archivos .ninja: 94
  Ejemplo: negative_datasets/dinner_party/dinner_party/training/chime6_train_u02_ch1_mmap/dtype.ninja

negative_datasets/no_speech
  Existe: True
  training: False
  validation: False
  testing: False
  Archivos .ninja: 188
  Ejemplo: negativ

In [14]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU disponibles:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.21.0
GPU disponibles: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [13]:
import yaml
from pathlib import Path

OUTPUT_NAME = "hola_sofia"

config = {
    "window_step_ms": 10,
    "train_dir": f"trained_models/{OUTPUT_NAME}",
    "features": [
        dict(features_dir="generated_augmented_features",
             sampling_weight=8.0, penalty_weight=2.0,
             truth=True, truncation_strategy="truncate_start", type="mmap"),

        dict(features_dir="negative_datasets/speech/speech",
             sampling_weight=10.0, penalty_weight=2.5,
             truth=False, truncation_strategy="random", type="mmap"),

        dict(features_dir="negative_datasets/dinner_party/dinner_party",
             sampling_weight=15.0, penalty_weight=3.0,
             truth=False, truncation_strategy="random", type="mmap"),

        dict(features_dir="negative_datasets/no_speech/no_speech",
             sampling_weight=5.0, penalty_weight=1.0,
             truth=False, truncation_strategy="random", type="mmap"),

        dict(features_dir="negative_datasets/dinner_party_eval/dinner_party_eval",
             sampling_weight=0.0, penalty_weight=1.0,
             truth=False, truncation_strategy="split", type="mmap"),

        dict(features_dir="confusable_features",
             sampling_weight=8.0, penalty_weight=5.0,
             truth=False, truncation_strategy="random", type="mmap"),
    ],
    "training_steps": [25000, 20000],
    "positive_class_weight": [2, 2],
    "negative_class_weight": [40, 50],
    "learning_rates": [0.001, 0.0001],
    "batch_size": 256,
    "time_mask_max_size": [5, 5],
    "time_mask_count": [1, 1],
    "freq_mask_max_size": [3, 3],
    "freq_mask_count": [1, 1],
    "eval_step_interval": 500,
    "clip_duration_ms": 1500,
    "target_minimization": 0.4,
    "minimization_metric": "ambient_false_positives_per_hour",
    "maximization_metric": "average_viable_recall",
}

Path(config["train_dir"]).mkdir(parents=True, exist_ok=True)

with open("/content/training_parameters.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("YAML generado correctamente")
print("Modelo:", OUTPUT_NAME)
print("Datasets:", len(config["features"]))
print("Pasos de entrenamiento:", sum(config["training_steps"]))

YAML generado correctamente
Modelo: hola_sofia
Datasets: 6
Pasos de entrenamiento: 45000


In [22]:
import shutil
from pathlib import Path

carpeta = Path("/content/trained_models/hola_sofia")

esperados = {
    "logs/train/events.out.tfevents.1791589976.21d49766a80d.31631.0.v2",
    "logs/validation/events.out.tfevents.1791589976.21d49766a80d.31631.1.v2",
    "model_summary.txt",
    "training_config.yaml",
}

actuales = {
    str(p.relative_to(carpeta))
    for p in carpeta.rglob("*")
    if p.is_file()
}

if actuales == esperados:
    shutil.rmtree(carpeta)
    print("Directorio de inicialización eliminado.")
    print("Listo para iniciar el entrenamiento.")
else:
    print("El contenido ha cambiado. No se ha eliminado nada.")
    print("Archivos actuales:", sorted(actuales))

Directorio de inicialización eliminado.
Listo para iniciar el entrenamiento.


In [48]:
from pathlib import Path
import shutil
from datetime import datetime

origen = Path("/content/trained_models/hola_sofia")
destino = Path("/content/drive/MyDrive/wakeword_training_hola_sofia/checkpoints")
destino.mkdir(parents=True, exist_ok=True)

fecha = datetime.now().strftime("%Y%m%d_%H%M%S")

for nombre in ["last_weights.weights.h5", "best_weights.weights.h5"]:
    archivo = origen / nombre

    if archivo.exists():
        copia = destino / f"{fecha}_{nombre}"
        shutil.copy2(archivo, copia)
        print(f"GUARDADO: {copia}")
        print(f"Tamaño: {copia.stat().st_size:,} bytes")
    else:
        print(f"NO EXISTE: {nombre}")

GUARDADO: /content/drive/MyDrive/wakeword_training_hola_sofia/checkpoints/20261010_004049_last_weights.weights.h5
Tamaño: 441,160 bytes
GUARDADO: /content/drive/MyDrive/wakeword_training_hola_sofia/checkpoints/20261010_004049_best_weights.weights.h5
Tamaño: 441,160 bytes


In [47]:
import os
import sys
import subprocess
from pathlib import Path

modelo = Path("/content/trained_models/hola_sofia")
pesos = modelo / "last_weights.weights.h5"

assert pesos.is_file(), "No se encuentran los pesos guardados"
assert Path("/content/training_parameters.yaml").is_file()

env = os.environ.copy()
env["PYTHONPATH"] = "/content/microWakeWord:" + env.get("PYTHONPATH", "")
env["XLA_FLAGS"] = "--xla_gpu_autotune_level=0"

cmd = [
    sys.executable, "-m", "microwakeword.model_train_eval",
    "--training_config", "/content/training_parameters.yaml",
    "--train", "1",
    "--restore_checkpoint", "1",
    "--test_tflite_streaming_quantized", "1",
    "--use_weights", "best_weights",
    "mixednet",
    "--pointwise_filters", "64,64,64,64",
    "--repeat_in_block", "1, 1, 1, 1",
    "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]",
    "--residual_connection", "0,0,0,0",
    "--first_conv_filters", "32",
    "--first_conv_kernel_size", "5",
    "--stride", "3",
]

print("Iniciando entrenamiento con los pesos del paso 500...", flush=True)

proc = subprocess.Popen(
    cmd,
    cwd="/content",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for linea in proc.stdout:
    print(linea, end="", flush=True)

proc.wait()
print("\nCódigo de salida:", proc.returncode)

Iniciando entrenamiento con los pesos del paso 500...
INFO:absl:Loading and analyzing data sets.
W0000 00:00:1791591416.632070   37944 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (256, 204, 40)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (256, 204, 1, 40) │          0 │ input_layer[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼──────────────────

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import shutil

origen = Path("/content/trained_models/hola_sofia")
destino = Path("/content/drive/MyDrive/wakeword_training_hola_sofia/checkpoints")

destino.mkdir(parents=True, exist_ok=True)

for nombre in ["last_weights.weights.h5", "best_weights.weights.h5"]:
    archivo = origen / nombre
    if archivo.exists():
        shutil.copy2(archivo, destino / nombre)
        print(f"Guardado: {nombre}")
    else:
        print(f"Todavía no existe: {nombre}")

In [ ]:
# === Export + push to Drive ===
import os, json, shutil, datetime

tflite_src = f'trained_models/{OUTPUT_NAME}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
assert os.path.exists(tflite_src), f'No model at {tflite_src}'

OUT_TFLITE = f'{OUTPUT_NAME}.tflite'
OUT_JSON = f'{OUTPUT_NAME}.json'
shutil.copy2(tflite_src, OUT_TFLITE)
print(f'wrote {OUT_TFLITE} ({os.path.getsize(OUT_TFLITE)/1024:.1f} KB)')

manifest = {
    'type': 'micro',
    'wake_word': WAKE_WORD,
    'author': AUTHOR,
    'website': AUTHOR_WEBSITE,
    'model': OUT_TFLITE,
    'trained_languages': TRAINED_LANGUAGES,
    'version': 2,
    'micro': {
        'probability_cutoff': PROBABILITY_CUTOFF,
        'feature_step_size': 10,
        'sliding_window_size': SLIDING_WINDOW_SIZE,
        'tensor_arena_size': TENSOR_ARENA_SIZE,
        'minimum_esphome_version': '2024.7.0',
    }
}
with open(OUT_JSON, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'wrote {OUT_JSON}')
print(json.dumps(manifest, indent=2))

for fn in (OUT_TFLITE, OUT_JSON):
    shutil.copy2(fn, f'{DRIVE_DIR}/{fn}')
    print(f'pushed {fn} -> {DRIVE_DIR}')

ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
with open(f'{DRIVE_DIR}/_run_finished.txt', 'w') as f:
    f.write(f'Training run finished at {ts}\n')
print()
print(f'DONE. Find your model at /content/drive/MyDrive/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite')


## Deploying to ESPHome devices

Drop the `.tflite` + `.json` next to your ESPHome YAML (e.g. in `/config/esphome/wakewords/<output_name>/`).

In your device YAML, replace your existing wake word:

```yaml
micro_wake_word:
  models:
    - model: wakewords/<output_name>/<output_name>.json
  on_wake_word_detected:
    - voice_assistant.start:
```

Then `esphome run <device>.yaml` (USB or OTA).

## Manifest tuning (likely needed)

The defaults work for **Hey Harold** specifically. Your model's confidence
distribution will differ. Iterate:

| Symptom | Knob |
|---|---|
| Doesn't fire on the wake word | Lower `probability_cutoff` (try 0.7, 0.6) |
| Fires on too many things | Raise `probability_cutoff` (try 0.92, 0.95) |
| LED fires but no STT response | `sliding_window_size: 5` (faster fire); also check Echo speaker mute |
| `Failed to allocate tensors` log | Raise `tensor_arena_size` (try 50000, 80000) |

## Known gotchas
- Editable install (`pip install -e ./microWakeWord`) requires kernel restart — broken for Run All
- `train.py` upstream calls `.numpy()` on numpy arrays under newer TF (this notebook patches it)
- T4 GPU OOMs during validation — use A100 + High-RAM
- Manifest path mismatch: training writes to `trained_models/<output_name>/`, manifest must match
- `voice_assistant` has no audio lookback; if MWW fires AFTER user speaks, STT gets silence
